# quantum-cortex, the external model arm on Colab (ADR-008 amendment 2026-09-16, addendum 2026-09-17)

The Molaison dissociation (the H.M. protocol: named after Henry Molaison, the patient whose hippocampus was removed in 1953 and who formed no new episodic memory from that day while keeping every skill he had; the protocol asks the model the same question: with its journal cut, do the episodes vanish while the skills stay?) run on an open model through the frozen prompt adapter. Nothing is trained.

**Runtime:** a GPU with bf16 (L4 or A100 on Colab Pro). The loader refuses a T4 (no bf16) and never changes the dtype. **Secrets (Colab, the key icon):** `GITHUB_TOKEN` (the repository is private), `QUANTUM_CORTEX_JOURNAL_KEY` (32 bytes hex; the journal is sealed with it). **Data:** the N1 slice `fineweb_edu_bytes.bin` for the skill arm, from Google Drive (cell 5).

**Cells:** 1 this header; 2 the clone; 3 the secrets and the model id; 4 the pinned revision resolved and the frame checked; 5 the data; 6 session A (plant, no model); 7 session B in a new process (the measurement, its log retained); 8 the repeat and the agreement; 9 the readout and the download. Run 2, 3 and 4 alone first and paste their outputs; then Runtime, Run all.


In [ ]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')                       # the repository is private
!rm -rf quantum-cortex
!git clone https://{token}@github.com/vaneeckhoutnicolas/quantum-cortex.git
%cd quantum-cortex
!pip -q install jsonschema cryptography transformers accelerate huggingface_hub


In [ ]:
# the key: from the Colab secret to the environment, never to a file; the model: one of the three pinned ids
import os
from google.colab import userdata
os.environ['QUANTUM_CORTEX_JOURNAL_KEY'] = userdata.get('QUANTUM_CORTEX_JOURNAL_KEY')
assert len(bytes.fromhex(os.environ['QUANTUM_CORTEX_JOURNAL_KEY'])) == 32, 'the key must be 32 bytes hex'
MODEL = 'Qwen/Qwen3-1.7B'          # then 'Qwen/Qwen3-4B', then 'Qwen/Qwen3-8B' (the addendum of 2026-09-17)
from cortex_c2b.external_arm import PINNED_MODELS, model_slug, frame_hash, build_frame
REV = PINNED_MODELS[MODEL]
SLUG = model_slug(MODEL, 'chat')
PH = frame_hash(build_frame())
print('journal key: present (not printed) | model', MODEL, '| revision', REV, '| slug', SLUG, '| prompt hash', PH)


In [ ]:
# the pinned revision against the hub's main, the card, the frozen frame against the retained file
!python -m cortex_c2b.external_arm --resolve {MODEL}
!nvidia-smi -L
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| bf16', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else None)
!python -m cortex_c2b.external_arm --frame
assert PH == '7fda43aced410d65', f'the frame hash moved: {PH} (a new hash is a new row, declared in the addendum first)'


In [ ]:
# the N1 slice for arm S: data/fineweb_edu_bytes.bin (the provenance is data/mixes/fineweb-edu-n1.provenance.json)
import os, shutil
DST = 'data/fineweb_edu_bytes.bin'
if not os.path.exists(DST):
    from google.colab import drive
    drive.mount('/content/drive')
    SRC = '/content/drive/MyDrive/quantum-cortex/fineweb_edu_bytes.bin'     # copied there once from the laptop
    assert os.path.exists(SRC), f'put the slice at {SRC} (the same file as C:\\git\\quantum-cortex\\data\\fineweb_edu_bytes.bin)'
    shutil.copy(SRC, DST)
print(DST, os.path.getsize(DST), 'bytes')


In [ ]:
# session A: the frozen protocol's facts planted through the gated write path into a journal sealed on disk (no model)
JDIR = f'runs/external-{SLUG}'
!rm -rf {JDIR}
!python -m cortex_c2b.external_arm --plant --journal {JDIR}


In [ ]:
# session B, a new process: the model on the three arms of arm E and on arm S; the log retained as a file
!mkdir -p metrics/mqar/logs
!python -m cortex_c2b.external_arm --probe --journal {JDIR} --model {MODEL} --revision {REV} --device cuda --out metrics/mqar 2>&1 | tee metrics/mqar/logs/external-arm-{SLUG}.log


In [ ]:
# the second execution, another process, and the agreement number for number
!python -m cortex_c2b.external_arm --probe --journal {JDIR} --model {MODEL} --revision {REV} --device cuda --out metrics/mqar --repeat 2>&1 | tee metrics/mqar/logs/external-arm-{SLUG}-repeat.log
!python -m cortex_c2b.external_arm --agree metrics/mqar/hm-lm-external-{SLUG}-{PH}.json metrics/mqar/hm-lm-external-{SLUG}-{PH}-repeat.json


In [ ]:
# the readout (a number read from a console is not a source: the files are), then the artefacts as one zip to download
import json, shutil
r = json.load(open(f'metrics/mqar/hm-lm-external-{SLUG}-{PH}.json'))
for k in ('verdict', 'hm_recall_on', 'hm_recall_off', 'hm_gap', 'hm_negctrl_rate', 'hm_invalid_citation_on', 'hm_valid_citation_on',
          'hm_false_abstention_on', 'hm_malformed_on', 'hm_skill_delta', 'hm_skill_delta_frame', 'hm_retrieval_hit', 'persistent', 'claimable'):
    print(f'{k:26s} {r[k]}')
shutil.make_archive(f'external-arm-{SLUG}', 'zip', root_dir='.', base_dir='metrics/mqar')
from google.colab import files
files.download(f'external-arm-{SLUG}.zip')
